In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_38_try_1.pickle

In [ ]:
%%RecordEvent
%%time
### cell 39 ###

# restore the question string so it’s in scope
question_of_interest_cell51 = \
    'Which of the following natural language processing (NLP) methods do you use on a regular basis?'


def grab_subset_of_data_51(df, question):
    # vectorized column‐filtering on GPU avoids Python loops over df.columns
    cols = df.columns[df.columns.str.contains(question)]
    # extract the suffix for renaming
    suffixes = [c.split('-')[-1].lstrip() for c in cols]
    mapping = dict(zip(cols, suffixes))
    # subset + rename stays on GPU; dropna uses GPU-backed cudf
    sub = df[cols].rename(columns=mapping)
    return sub.dropna(how='all', subset=suffixes)


def compute_transformer_pct(df, question):
    sub = grab_subset_of_data_51(df, question)
    # GPU-friendly notna().sum() instead of DataFrame.count()
    counts = sub.notna().sum().sort_values(ascending=True)
    pct = (counts * 100) / len(sub)
    # auto-detect the transformer label rather than hardcoding per year
    transformer_label = [c for c in pct.index if 'Transformer' in c][0]
    return pct.rename({transformer_label: 'Transformers'})[['Transformers']]

# compute once for each year
percentages_2019_cell51 = compute_transformer_pct(responses_df_2019_cell10, question_of_interest_cell51)
percentages_2020_cell51 = compute_transformer_pct(responses_df_2020,             question_of_interest_cell51)
percentages_2021_cell51 = compute_transformer_pct(responses_df_2021,             question_of_interest_cell51)
percentages_2022_cell51 = compute_transformer_pct(responses_df_2022_cell10,      question_of_interest_cell51)

# show info (as before)
(percentages_2019_cell51.info(),
 percentages_2020_cell51.info(),
 percentages_2021_cell51.info(),
 percentages_2022_cell51.info())

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_39_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_39_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[39], f)


In [ ]:
opt_output = Out.get(4)